# 02 — Liên kết MovieLens ↔ Freebase / Wikidata

Việc 15 (plan §12). Notebook trả lời ba câu hỏi: (1) bao nhiêu phim của MovieLens có liên kết sang Freebase, (2) liên kết
được tạo bằng quy tắc nào, (3) liên kết **sai** bao nhiêu (kiểm tay trên mẫu ngẫu nhiên 50 phim của ML-100K) và con số đó
đáng tin đến đâu.

**Chuỗi liên kết** (`src/tkgat/kg/linking.py`, script `scripts/03_link_entities.py`): ML-100K → ML-1M (tên + năm) →
ML-20M → Freebase / YAGO bằng bộ liên kết KB4Rec. Fact Wikidata được gắn vào phim qua mã Freebase (`P646`).

**Nguồn:** chỉ đọc `data/processed/links/linking_report.json`, `data/processed/links/ml100k_links.csv` và
`results/linking_manual_check*`. Mọi con số in ra từ các file này; không có số nào gõ tay.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from tkgat.analysis.report import sample_counts, wilson_interval

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LINKS = ROOT / "data" / "processed" / "links"
REPORT = pd.read_json(LINKS / "linking_report.json")
links = pd.read_csv(LINKS / "ml100k_links.csv")
sample = pd.read_csv(ROOT / "results" / "linking_manual_check_sample.csv", encoding="utf-8-sig")
counts = sample_counts(ROOT / "results" / "linking_manual_check_sample.csv")
print(counts)

## 1. Độ phủ: theo số phim và theo số rating

In [ ]:
rows = {}
for name in ("ml100k", "ml1m"):
    r = REPORT[name]
    rows[name.upper().replace("ML", "ML-")] = {
        "số phim": int(r["items"]),
        "phim có mã Freebase": int(r["freebase_items"]),
        "phủ theo phim": f"{r['freebase_item_share']:.1%}",
        "phủ theo rating": f"{r['freebase_rating_share']:.1%}",
        "phủ YAGO theo phim": f"{r['yago_item_share']:.1%}",
    }
display(pd.DataFrame(rows))

**Cách đọc.** Phủ theo *rating* cao hơn phủ theo *phim*: phim chưa có liên kết là phim ít người xem. Vì vậy phần rating
nằm trên phim không có KG rất nhỏ, và KG không bỏ sót phần lớn tín hiệu tương tác. Nhưng phim không liên kết cũng không
có fact nào, tức là KG kém đúng ở đuôi dài.

## 2. Quy tắc nào đã tạo ra liên kết (ML-100K)

In [ ]:
raw_rules = REPORT["ml100k"]["match_rules"]
unmatched = int(raw_rules.get("<NA>", 0))  # items that no rule could match to ML-1M
rules = pd.Series({k: v for k, v in raw_rules.items() if k != "<NA>"}).sort_values()
display(rules.to_frame("số phim"))
display(Markdown(f"Ngoài ra **{unmatched}** phim không ghép được sang ML-1M bằng quy tắc nào."))
fig, ax = plt.subplots(figsize=(7, 3), layout="constrained")
ax.barh(rules.index.astype(str), rules.values, color="#2a78d6")
ax.set_xlabel("số phim ML-100K")
ax.set_title("Quy tắc ghép ML-100K → ML-1M")
plt.show()
share = rules.get("title+year", 0) / rules.sum()
display(
    Markdown(
        f"`title+year` (tên và năm trùng) tạo ra **{share:.1%}** số phim đã ghép được; các quy tắc "
        "còn lại (tên khác, năm lệch 1, chứa chuỗi) lỏng hơn nên rủi ro sai cao hơn và đáng kiểm tay trước."
    )
)

## 3. Kiểm tay: 50 phim ngẫu nhiên

Mẫu 50 phim (seed 2019) ở `results/linking_manual_check_sample.csv`. Mỗi phim được kiểm bằng hai nguồn **độc lập với
quy tắc ghép**: năm phát hành trong Wikidata (từ `facts.tsv`) và tên trong YAGO. Phim nào cả hai nguồn khớp thì ghi `auto`;
phim còn lại (cột `needs_human_check = 1`) được Triết mở trang Wikidata để xem. Kết quả ghi ở cột `correct`.

In [ ]:
kinds = sample["auto_check"].value_counts().rename("số phim").to_frame()
display(kinds)
by_rule = (
    sample.assign(sai=lambda d: d[[c for c in d.columns if c.startswith("correct")][0]].astype(int).eq(0))
    .groupby("rule")["sai"]
    .agg(mau="size", sai="sum")
)
display(by_rule)

In [ ]:
n, wrong = counts["n"], counts["n_wrong"]
low, high = wilson_interval(wrong, n)
display(
    Markdown(
        f"**{wrong}/{n} liên kết sai** trong mẫu (kiểm tay {counts['n_human']} phim, "
        f"{n - counts['n_human']} phim có hai nguồn độc lập cùng khớp). "
        f"Khoảng tin cậy Wilson 95%: **[{low:.1%}; {high:.1%}]**."
    )
)

**Đọc con số này thế nào.** 0/50 sai *không* có nghĩa là tỷ lệ lỗi bằng 0. Với 50 phim, cận trên Wilson 95% là khoảng 7%:
tỷ lệ lỗi thật có thể đến mức đó mà mẫu vẫn không bắt được lỗi nào. Hai giới hạn nữa cần nói rõ:

1. **Kiểm tự động không hoàn toàn độc lập.** Năm Wikidata khớp nghĩa là phim có đúng năm đó, nhưng hai phim khác nhau
   cùng năm vẫn qua được; tên YAGO bù lại một phần. Vì vậy các phim `auto` chỉ được coi là "rất có khả năng đúng".
2. **Lỗi đã biết ngoài mẫu.** Trong lúc rà soát, nhóm đã gặp một lỗi của bộ KB4Rec trên ML-1M: phim *Clubland* có mã YAGO
   của một công viên bang (cột `yago_id` bên dưới). Nó không nằm trong mẫu của ML-100K, và là ví dụ cho thấy lỗi có tồn tại
   (decision 0003).

In [ ]:
links_1m = pd.read_csv(LINKS / "ml1m_links.csv")
hit = links_1m[links_1m["title"].fillna("").str.contains("clubland", case=False)]
display(hit if len(hit) else Markdown("*Không có phim tên 'Clubland' trong bảng liên kết ML-1M của lần dựng này.*"))

## 4. Hệ quả cho phần còn lại

- Item không liên kết (phần còn lại sau độ phủ ở mục 1, khoảng 3% số phim ML-100K) không có fact nào trong KG; mô hình xử lý chúng như item không có thuộc tính.
- Nếu một liên kết sai, **mọi** fact của phim đó sai theo (đạo diễn, giải thưởng, ngày). Tác động lớn nhất là với phim phổ biến
  vì chúng nhiều fact. Vì vậy báo cáo ghi cận trên 7% chứ không ghi "0%".
- Nên mở rộng mẫu (ví dụ 200 phim, ưu tiên quy tắc lỏng và phim phổ biến) trước khi kết luận chắc về độ chính xác. Việc này
  để sau 31/10.